# N136 · 线段树与区间聚合

**目标：** 把结合律与单位元落实为可组合节点。

**先修：** N031, N063, N134。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 建树；点更新；半开区间；幺半群；结合不等于交换；非空与空查询。

**配套讲解来源：** [同名逐章意见](../../comment/136_segment_tree_monoids.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章和上一章树状数组解决的是同一类问题——"一边改数据一边问区间聚合值"——但线段树更通用：树状数组天生适合求和这类"怎么交换顺序都没差"的运算，线段树只要求运算满足结合律（即加括号的方式不影响结果），不要求交换律（元素顺序不能换）。所以求和、区间最小值、甚至字符串拼接（'ab'+'cd' 和 'cd'+'ab' 可不一样！）都能塞进同一套代码。

一个具体到数字的例子（“运行示例”部分 用的）：把字符串 `'abcdef'` 建成线段树，聚合运算是拼接、单位元是空串，那么 `query(1, 5)` 返回 `'bcde'`——下标 1 到 4 这一段按原顺序拼起来。为什么顺序必须保证？因为如果代码里把右侧块用"往后追加"而不是"向前插入"的方式收集，查询 `(0, 5)` 就可能拼出 `'eabcd'` 这种乱序结果，而求和版本因为 3+5 和 5+3 相等，测试照样全过——这正是本章反复强调的隐蔽坑。

## 2. 基础知识：先读懂这些词

- **线段树（segment tree）**：一棵用数组存的近似完全二叉树，每个节点保存原数组某一段区间的聚合值（和 / 最小值 / 拼接结果……）。叶子存单元素，父节点存两个孩子的合并。你要它是因为改一个叶子只需要重算 log n 个祖先，查一段区间也只需要拼 log n 个块。
- **结合律（associativity）**：`(a∘b)∘c == a∘(b∘c)`，也就是"先合并谁"不影响结果。求和、求最小、字符串拼接都满足。线段树把大区间拆成小块分别聚合再拼回去，靠的就是它。
- **单位元（identity）**：和任何元素 x 合并后结果仍是 x 的"中性值"。加法的单位元是 0（x+0=x），拼接的是空串 `''`（x+''=x），最小值的是正无穷。空查询、补齐用的空叶子，全靠它撑着不出错。
- **幺半群（monoid）**：数学名词，指"一个满足结合律的运算 + 一个单位元"这套组合。本章的 `SegmentTree(values, combine, identity)` 构造参数就是这个抽象：你把幺半群的三件套（数据、运算、单位元）递进去，树就能工作。
- **结合不等于交换**：结合律说"括号随便挪"，交换律说"元素随便换"。拼接满足前者不满足后者，所以收集块的顺序必须严格保持原数组的从左到右。
- **半开区间**：`query(left, right)` 查的是下标 `[left, right)`，包含 left 不包含 right；`left == right` 是合法的空区间，结果应该是单位元。
- **非空与空查询**：空查询返回单位元（求和得 0、拼接得空串），这是幺半群接口的自然约定，测试里专门有一条检查它。

把 “思路推导”部分 提到的三个合格运算摆在一起对照，"幺半群"就具体了：

| 运算 | combine 写法 | identity | 满足结合律 | 满足交换律 |
|---|---|---|---|---|
| 求和 | `operator.add` | 0 | 是 | 是 |
| 最小值 | `min` | `float('inf')` | 是 | 是 |
| 字符串拼接 | `lambda a,b: a+b` | `''` | 是 | **否**（'ab'+'cd' ≠ 'cd'+'ab'） |

本章线段树对这三行全都适用，因为它的查询逻辑只依赖前两列（结合律 + 单位元），对最后一列不做假设——第三行能跑通，恰恰证明实现尊重了运算顺序。

## 3. 思路推导：从小例子开始

- **Step 1**：把叶子铺在数组后半段。先把 `size` 取成不小于 n 的 2 的幂（n=6 时 size=8），总长 `2*size`，叶子放下标 `size..size+n-1`，不足的叶子填单位元。
- **Step 2（建树）**：从 `i = size-1` 倒着到 1，逐个算 `tree[i] = combine(tree[2i], tree[2i+1])`。倒序保证算父亲时两个孩子都已就绪，一遍扫完，共 n-1 次合并。
- **Step 3（点更新 set）**：改下标 i 的值，就改叶子 `i+size`，然后一路 `i //= 2` 爬到根，每爬一层用两个孩子重新合并自己。爬过的节点恰好是且只是覆盖位置 i 的那些。
- **Step 4（区间查询）**：把 `[left, right)` 平移到叶子层（两端各加 size），然后用两个游标从下往上爬：左游标若落在奇数位（说明它是一个"右孩子"，自己只是半块），就把它的值收进左累加器 a、游标右移一位；右游标若落在奇数位，先减一，再把新位置的值"向前插入"右累加器 b；最后两边同时除以 2 上到父层。循环到两游标相遇，答案就是 `combine(a, b)`。
- **Step 5**：左累加器按"从左到右追加"，右累加器按"向前插入"（新块放在已收集内容的前面），这样 a 恒为答案的前缀、b 恒为答案的后缀，`combine(a,b)` 拼回去正好是原顺序。

**手算演示（“运行示例”部分 建树，'abcdef'）**：size=8，叶子层 `tree[8..15]` 依次是 `a,b,c,d,e,f,'',''`。自底向上合并：

| 内部节点 i | 合并自 | 结果 |
|---|---|---|
| 4 | tree[8]+tree[9] | 'ab' |
| 5 | tree[10]+tree[11] | 'cd' |
| 6 | tree[12]+tree[13] | 'ef' |
| 7 | tree[14]+tree[15] | ''（两片补齐空叶） |
| 2 | tree[4]+tree[5] | 'abcd' |
| 3 | tree[6]+tree[7] | 'ef'（右边空叶不捣乱，靠单位元） |
| 1 | tree[2]+tree[3] | 'abcdef' |

**再手算 query(1, 5)**：left=1+8=9，right=5+8=13，a=b=''。
第一轮：9 是奇数 → a = ''+'b' = 'b'，left=10；13 是奇数 → right=12，b = tree[12]+b = 'e'；left=10//2=5，right=12//2=6。
第二轮：5<6，5 是奇数 → a = 'b'+tree[5] = 'b'+'cd' = 'bcd'，left=6；6 是偶数，右不动；left=3，right=3，相遇停止。
返回 combine('bcd', 'e') = 'bcde'——注意 a 收的是 'b' 然后 'cd'（从左到右追加），b 收的 'e' 在最后，顺序完全正确。若把 b 改成向后追加，查询 (0,5) 时会先收 'e' 再收 'abcd'，拼出 'eabcd'，就乱了。

**再手算一次点更新 set(1, 'z')**（把 'abcdef' 的第二个字符换成 'z'，数组变 'azcdef'）：
1. 下标 1 对应叶子 `i = 1+8 = 9`，令 `tree[9] = 'z'`。
2. `i //= 2` → 4：`tree[4] = tree[8] + tree[9] = 'a'+'z' = 'az'`（原来是 'ab'，被重算）。
3. `i //= 2` → 2：`tree[2] = tree[4] + tree[5] = 'az'+'cd' = 'azcd'`。
4. `i //= 2` → 1：`tree[1] = tree[2] + tree[3] = 'azcd'+'ef' = 'azcdef'`。
5. `i //= 2` → 0，循环条件 `i>1` 不成立，停止。
一路上碰的节点是 9 → 4 → 2 → 1，恰好只有这四个节点的区间覆盖下标 1，其他节点一个都不用动——这就是"点更新 O(log n)"的直观含义。

## 4. 核心代码：SegmentTree

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
class SegmentTree:

    def __init__(self, values, combine, identity):
        self.n = len(values)
        self.combine = combine
        self.identity = identity
        self.size = 1
        while self.size < self.n:
            self.size *= 2
        self.tree = [identity] * (2 * self.size)
        self.tree[self.size:self.size + self.n] = list(values)
        for i in range(self.size - 1, 0, -1):
            self.tree[i] = combine(self.tree[2 * i], self.tree[2 * i + 1])

    def set(self, i, value):
        if not 0 <= i < self.n:
            raise IndexError('point outside tree')
        i += self.size
        self.tree[i] = value
        while i > 1:
            i //= 2
            self.tree[i] = self.combine(self.tree[2 * i], self.tree[2 * i + 1])

    def query(self, left, right):
        if not 0 <= left <= right <= self.n:
            raise IndexError('invalid half-open range')
        left += self.size
        right += self.size
        a = b = self.identity
        while left < right:
            if left & 1:
                a = self.combine(a, self.tree[left])
                left += 1
            if right & 1:
                right -= 1
                b = self.combine(self.tree[right], b)
            left //= 2
            right //= 2
        return self.combine(a, b)
```

### 逐段读懂代码

“分段实现”部分 只有一个 SegmentTree 类，我们逐个方法看。

**构造函数 `__init__`**

```python
def __init__(self,values,combine,identity):
        self.n=len(values); self.combine=combine; self.identity=identity; self.size=1
        while self.size<self.n: self.size*=2
        self.tree=[identity]*(2*self.size)
        self.tree[self.size:self.size+self.n]=list(values)
        for i in range(self.size-1,0,-1): self.tree[i]=combine(self.tree[2*i],self.tree[2*i+1])
```

先记下元素个数 n、合并函数和单位元。`while self.size<self.n: self.size*=2` 把 size 翻倍到不小于 n 的 2 的幂（n=6 → size=8；注意 n=0 时循环不执行，size 停在 1，空树也能构造）。`self.tree=[identity]*(2*self.size)` 开出全部填单位元的数组，多出来的叶子天然是"空元素"。切片赋值 `tree[size:size+n]=list(values)` 把输入铺进叶子层（拷贝一份，避免和调用方共享引用）。最后一行从 `size-1` 倒序循环到 1（`range` 的上界 0 不含），每个内部节点用两个孩子合并，一遍 O(size) 完成。

**点更新 `set`**

```python
def set(self,i,value):
        if not 0<=i<self.n: raise IndexError('point outside tree')
        i+=self.size; self.tree[i]=value
        while i>1:
            i//=2; self.tree[i]=self.combine(self.tree[2*i],self.tree[2*i+1])
```

第一行挡非法下标（负数或 ≥n 都抛错）。`i+=self.size` 跳到对应叶子，直接赋值——注意这里是赋值不是增量，和上一章树状数组的 `add` 语义不同。然后 `while i>1` 一路整除 2 爬向根，每层重算当前节点的聚合。`i>1` 保证根（下标 1）也会被更新到，且不会越界去碰 0 号。

**区间查询 `query`**

```python
def query(self,left,right):
        if not 0<=left<=right<=self.n: raise IndexError('invalid half-open range')
        left+=self.size; right+=self.size; a=b=self.identity
        while left<right:
            if left&1: a=self.combine(a,self.tree[left]); left+=1
            if right&1: right-=1; b=self.combine(self.tree[right],b)
            left//=2; right//=2
        return self.combine(a,b)
```

第一行检查半开区间合法性：left、right 都在 `[0, n]` 内且不倒序，`left==right`（空区间）放行。两个游标平移到叶子层后开始爬。`if left&1`（left 是奇数，即左孩子）说明当前块只有右半属于查询区间，收下它、left+=1 移到下一个块；`a=self.combine(a,...)` 是向右追加。`if right&1`（right 是奇数）说明右端左边那格属于区间，right-=1 后收 `tree[right]`；`b=self.combine(self.tree[right],b)` 把新块放在 b 前面（向前插入）——这就是保证非交换运算顺序正确的关键一行。最后两游标整除 2 同时升层，`left<right` 失效时收完，`combine(a,b)` 把前缀和后缀按原顺序拼成答案。

## 5. 分段实现：按顺序运行

**本章接口：** `SegmentTree(values, combine, identity)`、`SegmentTree.set(i, value)`、`SegmentTree.query(left, right)`

### SegmentTree

In [1]:
class SegmentTree:

    def __init__(self, values, combine, identity):
        self.n = len(values)
        self.combine = combine
        self.identity = identity
        self.size = 1
        while self.size < self.n:
            self.size *= 2
        self.tree = [identity] * (2 * self.size)
        self.tree[self.size:self.size + self.n] = list(values)
        for i in range(self.size - 1, 0, -1):
            self.tree[i] = combine(self.tree[2 * i], self.tree[2 * i + 1])

    def set(self, i, value):
        if not 0 <= i < self.n:
            raise IndexError('point outside tree')
        i += self.size
        self.tree[i] = value
        while i > 1:
            i //= 2
            self.tree[i] = self.combine(self.tree[2 * i], self.tree[2 * i + 1])

    def query(self, left, right):
        if not 0 <= left <= right <= self.n:
            raise IndexError('invalid half-open range')
        left += self.size
        right += self.size
        a = b = self.identity
        while left < right:
            if left & 1:
                a = self.combine(a, self.tree[left])
                left += 1
            if right & 1:
                right -= 1
                b = self.combine(self.tree[right], b)
            left //= 2
            right //= 2
        return self.combine(a, b)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

tree=SegmentTree(list('abcdef'),lambda a,b:a+b,'')
show_table(['半开区间','聚合字符串'],[((l,r),tree.query(l,r)) for l,r in [(0,6),(1,5),(2,2),(5,6)]])
show_table(['内部节点','节点聚合'],list(enumerate(tree.tree[1:],1)))

半开区间,聚合字符串
"(0, 6)",abcdef
"(1, 5)",bcde
"(2, 2)",
"(5, 6)",f


内部节点,节点聚合
1,abcdef
2,abcd
3,ef
4,ab
5,cd
6,ef
7,
8,a
9,b
10,c


## 7. 正确性、适用条件与复杂度

每个节点保持其区间按原顺序的聚合。查询把区间拆成不重叠有序块；结合律允许改变括号，不能允许改变顺序。左右双累加器分别维护已取前缀与后缀，最终combine(a,b)恢复整体顺序。

**代价：** 建树O(n)次combine，更新与查询O(log n)次combine，空间O(n)。这些上界以combine为O(1)为前提；字符串拼接自身复制字符，真实字节代价不能忽略。identity应不可变，combine应无副作用。

### 展开理解

为什么每个节点的值是对的：建树时我们保证算父亲之前两个孩子都已经是"各自区间的完整聚合"（倒序循环保证了这一点），set 更新时我们重算爬过的每个祖先，所以任何时刻"节点 = 它管的区间按原顺序聚合"这个性质都成立，空位由单位元补齐不产生干扰。

为什么查询是对的：两游标夹住的范围始终恰好是"还没收下的那部分区间"。收块的规则是：整块属于区间就留给上一层合并（偶数位），半块就当场收走（奇数位）。于是每一层收走的块互不重叠、按出现顺序分别进入 a（前缀）和 b（后缀），相遇时区间恰好收完。结合律允许我们"块内先合、块间后合"（改变括号），而 a 追加、b 前插的两条规则保证块与块的相对顺序和原数组一致（不改顺序），所以最终 `combine(a,b)` 就是答案。

复杂度：建树做了 O(size) 次合并，size < 2n，所以是 O(n)。set 每爬一层做一次合并，层数约 log2(size)，即 O(log n)。query 每层最多收左右各一块，同样是 O(log n) 次合并。n = 10^5 时 log2(n) ≈ 17，一次更新或查询约十七八次合并，十万次操作约两百万次基本动作，一眨眼完成。注意一个前提：这些上界假设 combine 本身是 O(1)——字符串拼接虽然是"一次 combine"，但它要复制字符，真实代价和串长成正比，长串场景不能忽略。另外 notebook 提醒 identity 应是不可变对象、combine 应无副作用，否则树里多个位置共享同一个可变对象时会被意外改坏。

和上一章树状数组摆在一起比一比，选型会更清楚：

| 维度 | 树状数组（N135） | 线段树（本章） |
|---|---|---|
| 能换聚合运算吗 | 基本只能前缀可减的（和、计数） | 任意"结合律 + 单位元" |
| 更新语义 | 增量 add | 赋值 set |
| 单次操作 | O(log n)，常数更小 | O(log n)，略重 |
| 代码量 | 很短 | 较长 |

一句话：只要"区间和"，树状数组短平快；一旦聚合不是减法可逆的（min、拼接）或要赋值语义，就换本章线段树。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的核心是"数值 + 字符串"双轨随机对拍（`rng=Random(136)` 固定种子可复现）：

- 外层 `for n in range(1,25)`：从长度 1 到 24 全覆盖，包含最短的非空数组和较长的奇偶长度。
- 每轮建两棵树：`tree` 是整数求和版（combine 是 `operator.add`，identity 是 0），`text` 是字符串拼接版（identity 是空串）。双轨的目的就是笔记本推导说的那件事：求和是交换的，顺序错了它也发现不了；拼接不交换，顺序一错立刻露馅。
- 每轮做 70 次"先随机改一个位置（两棵树同步 set），再随机查一段区间"：断言 `tree.query(l,r)==sum(a[l:r])` 和 `text.query(l,r)==''.join(letters[l:r])`。区间随机生成时允许 `l==r`（空区间），隐式测试了"空查询返回单位元"。改和查交错进行，验证更新后所有受影响节点都被正确重算。
- `assert SegmentTree([],operator.add,0).query(0,0)==0`：纯边界测试，空数组也能建树、空查询返回单位元 0，验证 size 循环和区间检查在 n=0 时不崩。

这套测试为什么非要带一个字符串版？可以想成一个"照妖镜"：假如你写 query 时把右侧累加器写反了，求和树的表现毫无异常（加法怎么交换都一样），随机对拍几百轮也发现不了；只有遇到拼接这种"顺序敏感"的运算，反序 bug 才会当场现形。这个套路值得搬进你自己的数据结构测试里——测结构时用一个不满足交换律的运算当探针。

最后打印 `N136: 所有本章断言通过`。

In [3]:
import operator

from random import Random

rng = Random(136)

for n in range(1, 25):
    a = [rng.randrange(-9, 10) for _ in range(n)]
    tree = SegmentTree(a, operator.add, 0)
    letters = list('x' * n)
    text = SegmentTree(letters, operator.add, '')
    for _ in range(70):
        i = rng.randrange(n)
        a[i] = rng.randrange(-9, 10)
        tree.set(i, a[i])
        letters[i] = rng.choice('abc')
        text.set(i, letters[i])
        l = rng.randrange(n + 1)
        r = rng.randrange(l, n + 1)
        assert tree.query(l, r) == sum(a[l:r])
        assert text.query(l, r) == ''.join(letters[l:r])

assert SegmentTree([], operator.add, 0).query(0, 0) == 0

print('N136: 所有本章断言通过')

N136: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 同时支持sum/min/字符串拼接。

**练习 2：** 解释左右累积顺序对非交换操作的重要性。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1：同时支持 sum/min/字符串拼接。** 提示：这个类的构造函数本来就吃"三件套"，你只需要喂三种不同的 (combine, identity)：求和用 `(operator.add, 0)`；最小值用 `(min, float('inf'))`（想想为什么单位元必须是"和任何数比较都不改变结果"的值）；拼接用 `(lambda a,b: a+b, '')`。用同一份小数组（比如 `[3,1,4]` 和 `'abc'`）分别建树，手算每个区间的期望值，再逐段 query 对照。特别注意 min 版的空查询应返回 `inf` 而不是 0。

**练习 2：解释左右累积顺序对非交换操作的重要性。** 提示：做一次"破坏性实验"——把 query 里 `b=self.combine(self.tree[right],b)` 改成 `b=self.combine(b,self.tree[right])`（前插改后接），然后跑 “自动测试”部分 的测试。你会发现求和断言照样全过，拼接断言当场爆掉。找一个具体反例：用 'abcdef' 那棵树查 `(0,5)`，正确结果是 'abcde'，改坏后会得到 'eabcd' 之类的乱序串。把这两次运行的对比写进你的报告，就是"为什么右侧必须向前插入"的最好解释。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [4]:
# 完整实现：本单元格不依赖其他单元格。
class SegmentTree:

    def __init__(self, values, combine, identity):
        self.n = len(values)
        self.combine = combine
        self.identity = identity
        self.size = 1
        while self.size < self.n:
            self.size *= 2
        self.tree = [identity] * (2 * self.size)
        self.tree[self.size:self.size + self.n] = list(values)
        for i in range(self.size - 1, 0, -1):
            self.tree[i] = combine(self.tree[2 * i], self.tree[2 * i + 1])

    def set(self, i, value):
        if not 0 <= i < self.n:
            raise IndexError('point outside tree')
        i += self.size
        self.tree[i] = value
        while i > 1:
            i //= 2
            self.tree[i] = self.combine(self.tree[2 * i], self.tree[2 * i + 1])

    def query(self, left, right):
        if not 0 <= left <= right <= self.n:
            raise IndexError('invalid half-open range')
        left += self.size
        right += self.size
        a = b = self.identity
        while left < right:
            if left & 1:
                a = self.combine(a, self.tree[left])
                left += 1
            if right & 1:
                right -= 1
                b = self.combine(self.tree[right], b)
            left //= 2
            right //= 2
        return self.combine(a, b)

# 示例与回归测试
import operator

from random import Random

rng = Random(136)

for n in range(1, 25):
    a = [rng.randrange(-9, 10) for _ in range(n)]
    tree = SegmentTree(a, operator.add, 0)
    letters = list('x' * n)
    text = SegmentTree(letters, operator.add, '')
    for _ in range(70):
        i = rng.randrange(n)
        a[i] = rng.randrange(-9, 10)
        tree.set(i, a[i])
        letters[i] = rng.choice('abc')
        text.set(i, letters[i])
        l = rng.randrange(n + 1)
        r = rng.randrange(l, n + 1)
        assert tree.query(l, r) == sum(a[l:r])
        assert text.query(l, r) == ''.join(letters[l:r])

assert SegmentTree([], operator.add, 0).query(0, 0) == 0

print('N136: 所有本章断言通过')

N136: 所有本章断言通过


## 11. 代表题与迁移

- **307. Range Sum Query - Mutable（区域和检索 - 数组可修改）**：本章 canonical 题，用线段树版"点赋值 set + 区间和 query"求解；和上一章树状数组同题不同实现，正好对比两套结构的写法差异（树状数组代码更短，线段树能换聚合运算）。
- **2286. Booking Concert Tickets in Groups（联动售票）**：extension 题，练的是把"区间聚合"从求和换成别的运算（比如每行剩余座位的最大值）再配合二分定位，本章的"传什么 combine 就支持什么聚合"的泛化接口正是这类题的地基。

学完这章建议把三个"泛化点"串起来复习一遍：(1) 换 combine/identity 就换聚合语义（练习 1）；(2) 查询收集块的顺序必须尊重运算是否交换（练习 2）；(3) set 是赋值、树状数组的 add 是增量，两章语义相反，混用前先想清楚。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。